# Hierarchical Clustering

## Prerequisite

Student should know about:

+ Partition based clustering algorithm, _K_-Means

+ Basic of graph theory

## Learning Objective

After completing this chapter, students should be able to:

+ Explain the concepts of _Proximity Matrix_, _Nested Partition_, _Denogram_

+ Differentiate between _Hierarchical Clustering_ and _Partition Based Clustering_

+ Identify _Agglomerative_ vs _Divisive_ Clustering

+ Explain and implement _DIANA_ algorithm

## Introduction

In the previous chapter, you have learned K-Means and it's varients. In K-Means algorithm, you had to explicitly specify the number of clusters required. This can sometimes be difficult because we may not exactly know the number of clusters to begin with. To this rescue comes the hierarchical clustering. In hierarchical clustering, you do not need to specify the number of clusters because it does not generate a specific number of clusters rather it generates hierarchy of clusters. Furthermore, the final result of K-Means depends on the initialization of the centroids which in most case is a random process; hence, making the final result random. But, most of the time the result in hierarchical clustering is deterministic which is required in many applications.

In this chapter, you are going to learn about the hierarchical clustering and understand how it solves the problems of partition based clustering like *K-Means*. First off, let's learn some of the key terminologies that are important in hierarchical clustering.

### Proximity Matrix
Let us assume that we have following data points $D = \{x_1, x_2, x_3, x_4\}$. The proximity matrix given below tells us about the dissimilarity or distance between two data points.
$$
\mathbf{\mathcal D} = \begin{bmatrix}0 & 0.2 & 1.0 & 2\\
0.2 & 0 & 0.6 & 0.1\\
0.1 & 0.2 & 0 & 0.5\\
2 & 0.1 & 0.5 & 0
\end{bmatrix}
$$

The distance matrix $\mathbf{\mathcal D}$ for four data points is of size $4\times 4$. Also, $\mathbf{\mathcal D}_{i,j}$ denotes the distance between $x_i$ and $x_j$ given as,
$$
\mathbf{\mathcal D}_{i,j} = d(x_i, x_j)
$$

If $d(x_i, x_j) = d(x_j, x_i) \hspace{0.2cm} \forall_{i,j} \hspace{0.2cm}$(changing order of data points), then the distance matrix would be a symmetric matrix.

Also, the proximity matrix can have either the distance or the similarity between the data points. The similarity measure for example can be cosine similarity and the distance measure can be euclidian distance.

Finding the correct proximity metric for the problem at hand can be challenging and there are no shortcuts in doing so. In most of the cases, the domain knowledge is used to find an optimal proximity metric. In this lesson, we will mainly focus on the euclidian distance.

In the hierarchical clustering, only the proximity matrix is required for making the hierarchy of clusters; hence, we do not actually need to know how the data are distributed in the space. This can be helpful in many real world applications.

### Nested Partition

Before moving forward, let's understand about nested partition and why we need it. The partition or cluster in general are represented with a set. Let's imagine we have the following dataset,
$$
\mathbf{\mathcal D} = \{x_1, \dots, x_n\}
$$

Now, a partition $\mathcal A$ splits the data into the subsets $\{\mathcal A_1, \dots, \mathcal A_m\}$ that satisfies the following,
$$
\mathcal A_i \cap \mathcal A_j = \varnothing \hspace{0.2cm} \forall_{i \neq j}

Where $\cap$ represents intersection and $\pi$ is empty set. This equation means data points should belong to only one cluster.

and\\
\mathcal A_i \cup \dots \cup \mathcal A_m = \mathbf{\mathcal D}
$$

Here, $\cup$ represents the union set operation. This equation means by combining those disjoint sets $\mathcal A_i, \dots$, we can obtain the original set we began with.

Now, imagine a new partition $\mathcal B$ splits $\mathcal A$. For $\mathcal B$ to be nested in $\mathcal A$; we should be able to create the partitions of $\mathcal A$ by combining the partitions of $\mathcal B$. The combining operation refers to the union operation in this case.

__EXAMPLE:__

Let's say we have a sample dataset given as,
$$
\mathcal D = \{x_1, x_2, x_3, x_4, x_5, x_6, x_7\}
$$

Imagine the partition $\mathcal A$ splits it into,
$$
\mathcal A = \{(x_1, x_2, x_5, x_7), (x_3, x_4, x_6)\}
$$

As you can see, each partitions are disjoint and $\mathcal A$ contains all the element of the original set $\mathbf{\mathcal D}$.

With the next partition operation $\mathcal B$ in $\mathcal A$, we get the following result,
$$
\mathcal B = \{(x_1, x_5), (x_2, x_7), (x_3), (x_4, x_6)\}
$$

To determine if $\mathcal B$ is a nested partition or not, we can tey to combine the partitions in $\mathcal B$ in order to obtain $\mathcal A$.

In above example, partition $(x_1, x_2, x_5, x_7)$ can be created by combining the first and second partition of $\mathcal B$. Similarly, partition $(x_3, x_4, x_6)$ can be created by combining the third and fourth elements of $\mathcal B$; therefore, the partition $\mathcal B$ is a nested partition of $\mathcal A$.

Let's see an example where the new partition is not a nested partition. If we had splitted the partition $\mathcal A$ with $\mathcal B^\prime$ as follows,
$$
\mathcal B^\prime =\{(x_1, x_3), (x_2, x_5, x_7), (x_4, x_6)\}
$$

You can check that we would not be able to obtain our original partition $\mathcal A$ by merging the partitions of $\mathcal B^\prime$ making $\mathcal B^\prime$ not a nested partition.

Now, the nested partition comes in hierarchical clustering because the hierarchical clustering can be defined in terms of nested partition as follows:

'''A hierarchical clustering is a sequence of nested partition where each subsequent parititon is nested on the previous one.'''

So, in the above example, the sequence $\left[\mathcal D, \mathcal A, \mathcal B\right]$ is a hierarchical clustering, whereas $\left[\mathcal D, \mathcal A, \mathcal B^\prime\right]$ is not.

### Dendogram

A more convenient way of representing the hierarchical clustering is by using a tree like structure called _dendogram_. An example of dendogram is shown in the figure below:

<div align="center">
    <figure>
        <!-- <img src="https://doc.google.com/a/fusemachines.com/uc?export=download&id=1-9Lt0WqDc2nXYsOyjyGWC0mjPSofqsQP" height="400">
         -->
        <img src="https://i.postimg.cc/76HXbp3v/image.png" height="400">
        
        <figcaption>Figure 1: Dendogram with arrow showing agglomerative and divisive clustering.</figcaption>
    </figure>
</div>

In the above Figure, you can see that it is a binary tree representation where each node splits into two branches. Using the terminology of tree, we can say that the root node here represents the partition containing all the data points, each node represents a partition and finally the leaf nodes are singleton partition (partition containing only one element). Similarly, the height of each edge is proportional to the distance between two data points, two partitions or a data point and a partition.

In dendogram, the root node is just our dataset and the leaf node is just the data points, so they are of no use. Now, to get some required clusters in hierarchical clustering, we can cut the dendogram at any level shown by the dotted line parallel to $x$-axis.

One of the advantages of dendogram is that it gives an intuitive understanding of relationship between the clusters that can be very helpful in some application like taxonomy.

Now let's create a dendrogram using a sample data set .





<div align="center">
    <figure>
        <!-- <img src="https://doc.google.com/a/fusemachines.com/uc?export=download&id=1EpqpVA9VNCaUE0NkaUvEPYVHycvMuCnz" height="400"> -->
        <img src ="https://i.postimg.cc/KYRpNXKn/image.png" height="400">
<figcaption>Figure 2:Scatter Plot of data</figcaption>
    </figure>
</div>

Here, each data point is a cluster of its own. We want to determine a way to compute the distance between each of these points. For this, we try to find the shortest distance between any two data points to form a cluster. Once we find those with the least distance between them, we start grouping them together and forming clusters of multiple points.

This can be represented in  dendrogram as shown below.

<div align="center">
    <figure>
        <!-- <img src="https://doc.google.com/a/fusemachines.com/uc?export=download&id=1BH44Qd347VdbW42w_V9FpYl_cSnWSua2" height="400"> -->
        <img src ="https://i.postimg.cc/k5bzC3n7/image.png" height ="400">
<figcaption>Figure 3: First Dendrograph</figcaption>
    </figure>
</div>

Here we will have 3 groups: P1-P2, P3-P4, and P5-P6 their respective dendrograms can be constructed as below.


<div float='left'>
<div>
    <figure>
        <!-- <img src="https://doc.google.com/a/fusemachines.com/uc?export=download&id=1kdFfGmgmAsdsHVoykeiccG_2s7JDLGFf" height="400"> -->
        <img src="https://i.postimg.cc/vTTWRnnk/image.png" height="400">    
        <figcaption>Figure 4: Plot</figcaption>
    </figure>
</div>


<div>
    <figure>
        <!-- <img src="https://doc.google.com/a/fusemachines.com/uc?export=download&id=1UkF7sAIJBn8ZVpXmOwaRPvbTQUUzF-7Q" height="400"> -->
        <img src="https://i.postimg.cc/25xSf84f/image.png" height="400">
<figcaption>Figure 5: Dendrograph</figcaption>
    </figure>
</div>

</div>


Now we will bring two close groups together.
P3-P4 and P5-P6 are all under one dendrogram because they are together than P1-P2 group.



<div>
    <figure>
        <!-- <img src="https://doc.google.com/a/fusemachines.com/uc?export=download&id=1q5nhx_w0Cfnt0mZdpulvBx5NrlKxaqA7" height="400"> -->
        <img src="https://i.postimg.cc/DytrS5Rw/image.png" height ="400">
<figcaption>Figure 6: Plot</figcaption>
    </figure>
</div>




<div>
    <figure>
        <!-- <img src="https://doc.google.com/a/fusemachines.com/uc?export=download&id=1VhlhY5vIfIERw1QS0fgZoTWWxfas7AR2" height="400"> -->
        <img src ="https://i.postimg.cc/fTwLBdXf/image.png" height="400">
<figcaption>Figure 7: Dendrograph</figcaption>
    </figure>
</div>




Now we will bring another group of clusters together with the big cluster.
If we have multiple other clusters we merge another cluster to this one according to the shortest distance.




<div>
    <figure>
        <!-- <img src="https://doc.google.com/a/fusemachines.com/uc?export=download&id=1OYyQXe-HoPzUbIaRuc-az89r8tgfZ7cU" height="400"> -->
        <img src ="https://i.postimg.cc/1tSdFw7L/image.png" height="400">
<figcaption>Figure 8: Plot</figcaption>
    </figure>
</div>




<div>
    <figure>
        <!-- <img src="https://doc.google.com/a/fusemachines.com/uc?export=download&id=1hlcSqfTovt7r4gfPoJ7EsEjsdUqLu0VD" height="400"> -->
        <img src="https://i.postimg.cc/zXYjpKCB/image.png" height="400">
<figcaption>Figure 9: Dendrograph</figcaption>
    </figure>
</div>

In this way, we can find the dendrogram graph of the dataset.

### Types of Clustering

Hierarchical clustering can be divided into two parts given in the list below:

+ Agglomerative Clustering

>Agglomerative clustering, also known as bottom-up hierarchical clustering, is a clustering approach where we start from the bottom of the hierarchy/dendogram and build the partition up until a specified stopping condition. The upward arrow in Figure 1 shows the process of agglomerative clustering where we start from singleton partition and build up the tree until we reach the root node.

+ Divisive Clustering

> Divisive clustering is the opposite of agglomerative clustering in the sense that the dendogram is created in top to bottom fashion. Initially all the datapoints belongs to a single cluster, then we iteratively apply the split until a stopping criteria is reached or all the clusters are singleton. The downward arrow in Figure 1 shows the process of divisive clustering.

In this chapter, you are going to learn about divisive clustering and in next chapter you will learn about agglomerative clustering.

## Divisive Clustering

Divisive clustering algorithm is a top-down approach where we start at the top of the tree (root node) and create clusters by spliting a single cluster into two clusters.



The working of divisive hierarchical clustering algorithm is shown in the animation below.

<div align="center">
    <figure>
        <!-- <img src="https://doc.google.com/a/fusemachines.com/uc?export=download&id=1A1qfmbh_gPMekVdKpbeObYOORShO0rB6" alt="Divisive clustering for a toy dataset on the left and simultaneous representation with dendogram"> -->
        <img src="https://i.postimg.cc/7Zmdpm9X/image.png" alt="Divisive clustering for a toy dataset on the left and simultaneous representation with dendogram">
        <figcaption>Figure 7: Divisive clustering on a toy dataset. (a) Scatter plot with highlighted clusters animation. (b) Divisive clustering shown in dendogram animation</figcaption>
    </figure>
</div>

In the Figure 3(a), the circular boundary around the data represents a cluster. With divisive clustering, you can see in the scatter plot (figure (a)) that we start by assigning all the data points to a single cluster, then we iteratively split the cluster to form smaller clusters until we reach the stopping condition or we obtain all the singleton cluster. In the dendogram plot in Figure 7(b), contrary to agglomerative clustering, we start from the root node of the binary tree and start the branch formation from the top to bottom fashion until we achieve all the leaf nodes as a singleton partitions.

### General Algorithm

A general divisive hierarchical clustering algorithm is given below,

<br>

---
__ALGORITHM__: General Divisive Hierarchical Clustering

---
Start with root node containing all the data points

__REPEAT:__
>
> Split parent cluster $C$ into two parts $C_a$ and $C_b$ using a split method
>
> Choose the next cluster from non-singleton leaf clusters with maximum value of squared error.
>
__UNTIL:__ All the leaves are singleton clusters

---

In the above algorithm, we talked about the splitting method but we did not go into the details. Also, we choose the next cluster using maximum squared error which is not the only way to do so. Here, we will discuss about some of the divisive method and criterion and also the method of choosing the next cluster for the split.
>
>+ Split method and criterion
>
> We should consider how are we going to split a given cluster into smaller ones. We can use Bisecting _K_-Means algorithm for this purpose. Also, if the data are categorical we can use the Gini-index.
>
>+ Choosing the cluster to split
>
> Choosing which cluster to split next can also be important when we are trying to achieve good results early on in the clustering.
>

We are going to cover DIANA divisive clustering algorithm in the following section in detail.

### DIANA Algorithm

DIANA (DIvisive ANAlysis) is a heuristic method for divisive clustering.


Let's imagine we are trying to split the cluster $\mathcal C$ into two clusters $\mathcal C_a$ and $\mathcal C_b$. The DIANA algorithm for this purpose is given below:

---
__ALGORITHM:__ DIvisive ANAlysis (DIANA)

---
1. Assign all data of $C$ to $C_a$ and $C_b$ to an empty cluster.

2. For each data point in $C_a$, i.e $\mathbf x_i \in C_a$, do

    a. Compute the average distance of $x_i$ to all other data points as,
$$
d(\mathbf x_i, C_a\backslash\{\mathbf x_i\}) = \frac{1}{N_{c_a} - 1} \sum_{\mathbf x_j \in C_a\\ \ \ i\neq j}d(\mathbf x_i, \mathbf x_j)
$$

    b. For the remaining iteration, compute the difference between the average distance to $C_a$ and $C_b$ as,
$$
\begin{align}
D_i &= d(\mathbf x_i, C_a\backslash\{\mathbf x_i\}) - d(\mathbf x_i, C_b)\\
&= \frac{1}{N_{C_a} - 1} \sum_{\mathbf x_j \in C_a\\ \ \ i \neq j} d(\mathbf x_i, x_j) - \frac{1}{N_{C_b}} \sum_{\mathbf x_k \in C_b} d(\mathbf x_i, \mathbf x_k)
\end{align}
$$

3. a. For first iteration, add the data point $x_i$ with the maximum $D_i$ to the empty cluster $C_b$

    b.For iterations except first

    __IF:__ maximum value of $D$ is greater than zero given as,
    $$D_{max} = \max(D_1, D_2, \dots) \gt 0$$
    move the data point $x_{max}$ corresponding to $D_{max}$ to $C_b$

    Repeat Step 2(b) and 3(b)

    __OTHERWISE:__ stop
---

With this algorithm you will be able to cluster any given data points according to the divisive clustering approach.

## Key Takeaways

+ Proximity Matrix gives the nearness or distance between any two data points

+ Hierarchical clustering gives multiple clusters arranged in hierarchy or in terms of nested partition, it gives a list of nested partititons

+ Dendogram can be very useful when visualizing and understanding the clustering

+ Divisive divides a single big clusters into small ones from the top down fashion

+ DIANA is one of the popular divisive algorithm where the division is done based on the closeness to two choosen points from the cluster